# Merge Keywords One Sheet

Merges result files from several related search keywords (e.g. a parent keyword and its sub-keywords) into one dataset: one representative row per ASIN, de-duplicated price history, and a column recording every keyword under which each ASIN appeared.

**Setup.** Data paths point to `/content/drive/MyDrive/YOUR_PROJECT_FOLDER/...` — change `YOUR_PROJECT_FOLDER` to your own folder before running. API keys are read from Colab Secrets / environment variables, never hard-coded.

> Code comments and console messages are in Korean (original working language). See the [folder README](README.md) for the pipeline overview and the [disclaimer](../README.md#disclaimer).


In [ ]:
# ================================================================================
# Amazon Patch 제품 가격이력 통합
#
# 검색어 구조:
#
#       patch
#         │
#     skin patch
#    /    |     \
# acne  wrinkle  eye
# patch   patch   patch
#
# 핵심:
# 1. 제품분석: ASIN 기준 1개 대표 행
# 2. 가격요약: ASIN 기준 1개 대표 행
# 3. 가격이력: ASIN + 날짜/시간 + 가격 기준으로 중복 제거
#    → 날짜가 다르면 모두 유지
# 4. 같은 ASIN이 여러 검색어에서 발견되면 검색어 관계를 반영
# 5. "검색키워드_전체"에 해당 ASIN이 발견된 모든 검색어 기록
# 6. eye patch는 skin patch의 하위 검색어로 처리
# ================================================================================

import pandas as pd
import numpy as np
import os
import re
from datetime import datetime


# ================================================================================
# 0. 파일 선택
# ================================================================================

from google.colab import files

print("=" * 100)
print("Amazon Patch 파일 선택")
print("=" * 100)

print("""
아래 5개 파일을 선택해주세요.

  1. patch
  2. skin patch
  3. acne patch
  4. wrinkle patch
  5. eye patch

파일명 예시:
  patch.xlsx
  skin_patch.xlsx
  acne_patch.xlsx
  wrinkle_patch.xlsx
  eye_patch.xlsx

여러 파일을 한 번에 선택해도 됩니다.
""")

uploaded = files.upload()


# ================================================================================
# 업로드된 파일 → 검색키워드 자동 매칭
# ================================================================================

keyword_patterns = {

    "patch": [
        "patch"
    ],

    "skin patch": [
        "skin patch",
        "skin_patch"
    ],

    "acne patch": [
        "acne patch",
        "acne_patch"
    ],

    "wrinkle patch": [
        "wrinkle patch",
        "wrinkle_patch"
    ],

    "eye patch": [
        "eye patch",
        "eye_patch",
        "eye+patch"
    ],
}


def detect_keyword(filename):

    """
    파일명에서 검색키워드를 자동으로 찾는다.

    예:
        patch.xlsx
            → patch

        skin_patch.xlsx
            → skin patch

        acne_patch.xlsx
            → acne patch

        wrinkle_patch.xlsx
            → wrinkle patch

        eye_patch.xlsx
            → eye patch

        eye+patch.xlsx
            → eye patch
    """

    name = os.path.splitext(
        filename
    )[0].lower()

    # --------------------------------------------------------------------------
    # 중요:
    # 구체적인 검색어를 먼저 검사
    #
    # 예:
    # skin_patch.xlsx
    # → skin patch를 먼저 찾아야 함
    #
    # eye_patch.xlsx
    # → eye patch를 먼저 찾아야 함
    #
    # 그렇지 않으면 전부 "patch"로 인식될 수 있음.
    # --------------------------------------------------------------------------

    search_order = [

        "wrinkle patch",
        "wrinkle_patch",

        "acne patch",
        "acne_patch",

        "skin patch",
        "skin_patch",

        "eye patch",
        "eye_patch",
        "eye+patch",

        "patch",
    ]

    for pattern in search_order:

        if pattern in name:

            return pattern.replace(
                "_",
                " "
            ).replace(
                "+",
                " "
            )

    return None


# ================================================================================
# 파일별 검색키워드 매칭
# ================================================================================

files = {}

print("\n" + "=" * 100)
print("파일 매칭 결과")
print("=" * 100)

for filename in uploaded.keys():

    keyword = detect_keyword(
        filename
    )

    if keyword is None:

        print(
            f"⚠️ 검색키워드 인식 실패: "
            f"{filename}"
        )

        continue

    # 동일 검색어 파일이 여러 개라면
    # 마지막 파일로 덮어쓰지 않음
    if keyword in files:

        print(
            f"⚠️ {keyword} 파일이 이미 존재합니다."
        )

        print(
            f"   기존: {files[keyword]}"
        )

        print(
            f"   신규: /content/{filename}"
        )

        continue

    files[keyword] = (
        f"/content/{filename}"
    )

    print(
        f"✓ {keyword:<15} ← {filename}"
    )


# ================================================================================
# 필수 파일 확인
# ================================================================================

required_keywords = [

    "patch",
    "skin patch",
    "acne patch",
    "wrinkle patch",
    "eye patch",

]


missing = [
    k
    for k in required_keywords
    if k not in files
]


if missing:

    print("\n" + "=" * 100)
    print("⚠️ 파일 확인 필요")
    print("=" * 100)

    print(
        "찾지 못한 파일:",
        ", ".join(missing)
    )

    print(
        "\n파일명이 위 키워드를 포함하는지 확인해주세요."
    )

else:

    print("\n" + "=" * 100)
    print("✅ 5개 파일 모두 확인")
    print("=" * 100)

    for keyword in required_keywords:

        print(
            f"{keyword:<15} → "
            f"{files[keyword]}"
        )


# ================================================================================
# 출력 파일
# ================================================================================

timestamp = datetime.now().strftime(
    "%Y%m%d_%H%M%S"
)

output_file = (
    f"/content/Amazon_patch_5keyword_append_{timestamp}.xlsx"
)

print(
    f"\n출력 파일:\n{output_file}"
)


# ================================================================================
# 1. 검색어 계층
# ================================================================================
#
# 구조:
#
# patch
# └── skin patch
#     ├── acne patch
#     ├── wrinkle patch
#     └── eye patch
#
# 낮은 숫자 = 상위/넓은 범위
# 높은 숫자 = 하위/구체적인 범위
#
# ================================================================================

keyword_level = {

    "patch": 1,

    "skin patch": 2,

    "acne patch": 3,
    "wrinkle patch": 3,
    "eye patch": 3,

}


# ================================================================================
# 2. 검색어 대표값 선택 규칙
# ================================================================================

def choose_representative_keyword(keywords):

    """
    같은 ASIN이 여러 검색어에서 발견됐을 때
    제품분석 / 가격요약에서 사용할 대표 검색어를 결정한다.

    검색어 hierarchy:

        patch
          ↓
        skin patch
        /     |      \
    acne   wrinkle   eye
    patch   patch    patch


    대표 검색어 규칙:

    ① patch만 발견
       → patch

    ② patch + skin patch
       → skin patch

    ③ patch + acne patch
       → acne patch

    ④ patch + wrinkle patch
       → wrinkle patch

    ⑤ patch + eye patch
       → eye patch

    ⑥ skin patch + acne patch
       → skin patch

    ⑦ skin patch + wrinkle patch
       → skin patch

    ⑧ skin patch + eye patch
       → skin patch

    ⑨ acne patch + wrinkle patch
       → 먼저 append된 검색어

    ⑩ acne patch + eye patch
       → 먼저 append된 검색어

    ⑪ wrinkle patch + eye patch
       → 먼저 append된 검색어

    ⑫ skin patch + 세부 검색어
       → skin patch

    ⑬ 다섯 검색어 모두 발견
       → skin patch

    검색키워드_전체에는
    발견된 모든 검색어를 유지한다.
    """

    keywords = [

        str(k).strip().lower()

        for k in keywords

        if pd.notna(k)
        and str(k).strip()

    ]


    # --------------------------------------------------------------------------
    # 중복 제거
    # 입력 순서 유지
    # --------------------------------------------------------------------------

    unique_keywords = list(
        dict.fromkeys(
            keywords
        )
    )


    if not unique_keywords:

        return "", []


    keyword_set = set(
        unique_keywords
    )


    # --------------------------------------------------------------------------
    # 1) skin patch
    # --------------------------------------------------------------------------
    #
    # skin patch는 acne / wrinkle / eye를 모두 포함하는
    # 상위 범주이므로 가장 우선.
    #

    if "skin patch" in keyword_set:

        return (
            "skin patch",
            unique_keywords
        )


    # --------------------------------------------------------------------------
    # 2) skin patch가 없는 경우
    #
    # 구체적인 검색어 중 먼저 append된 검색어 사용
    #
    # 예:
    #
    # acne patch + wrinkle patch
    # → 먼저 들어온 검색어
    #
    # eye patch + wrinkle patch
    # → 먼저 들어온 검색어
    #
    # --------------------------------------------------------------------------

    specific_keywords = [

        k
        for k in unique_keywords

        if k in [
            "acne patch",
            "wrinkle patch",
            "eye patch"
        ]

    ]


    if specific_keywords:

        return (
            specific_keywords[0],
            unique_keywords
        )


    # --------------------------------------------------------------------------
    # 3) patch
    # --------------------------------------------------------------------------

    if "patch" in keyword_set:

        return (
            "patch",
            unique_keywords
        )


    # --------------------------------------------------------------------------
    # 4) 예상하지 못한 검색어
    # --------------------------------------------------------------------------

    return (
        unique_keywords[0],
        unique_keywords
    )


# ================================================================================
# 3. 유틸 함수
# ================================================================================

def normalize_asin(x):

    if pd.isna(x):

        return ""

    x = str(x).strip().upper()


    # Amazon URL

    m = re.search(
        r"/(?:dp|gp/product)/([A-Z0-9]{10})",
        x
    )

    if m:

        return m.group(1)


    # 일반 ASIN

    m = re.search(
        r"\b([A-Z0-9]{10})\b",
        x
    )

    if m:

        return m.group(1)


    return x


def clean_columns(df):

    df = df.copy()

    df.columns = [

        str(c).strip()

        for c in df.columns

    ]

    return df


def ensure_column(
    df,
    col,
    default=np.nan
):

    if col not in df.columns:

        df[col] = default

    return df


def normalize_keyword(x):

    if pd.isna(x):

        return ""

    return (
        str(x)
        .strip()
        .lower()
    )


# ================================================================================
# 4. Excel 읽기
# ================================================================================

data = {}

print("=" * 100)
print("Amazon Patch 통합 시작")
print("=" * 100)


for keyword, filepath in files.items():

    print("\n" + "-" * 100)

    print(
        f"[{keyword}]"
    )

    print(filepath)


    if not os.path.exists(filepath):

        print("❌ 파일 없음")

        continue


    try:

        xl = pd.ExcelFile(
            filepath
        )

        print(
            "시트:",
            xl.sheet_names
        )


        data[keyword] = {}


        for sheet in [

            "제품분석",
            "가격요약",
            "가격이력"

        ]:

            if sheet not in xl.sheet_names:

                print(
                    f"⚠️ {sheet} 없음"
                )

                continue


            df = pd.read_excel(
                filepath,
                sheet_name=sheet
            )

            df = clean_columns(
                df
            )


            data[keyword][sheet] = df


            print(
                f"{sheet}: "
                f"{len(df):,} rows"
            )


    except Exception as e:

        print(
            f"❌ 파일 읽기 오류: {e}"
        )


# ================================================================================
# 5. 제품분석 통합
# ================================================================================

print("\n" + "=" * 100)
print("① 제품분석 통합")
print("=" * 100)


product_frames = []


for keyword, sheets in data.items():

    if "제품분석" not in sheets:

        continue


    df = sheets[
        "제품분석"
    ].copy()


    df = ensure_column(
        df,
        "검색키워드"
    )


    df = ensure_column(
        df,
        "ASIN"
    )


    # --------------------------------------------------------------------------
    # 검색키워드
    # --------------------------------------------------------------------------

    df["검색키워드"] = (

        df["검색키워드"]

        .replace(
            [
                "",
                "nan",
                "None"
            ],
            np.nan
        )

        .fillna(
            keyword
        )

        .apply(
            normalize_keyword
        )

    )


    # --------------------------------------------------------------------------
    # ASIN
    # --------------------------------------------------------------------------

    df["ASIN"] = (

        df["ASIN"]

        .apply(
            normalize_asin
        )

    )


    # --------------------------------------------------------------------------
    # 원본 파일 검색어 기록
    # --------------------------------------------------------------------------

    df["_source_keyword"] = keyword


    product_frames.append(
        df
    )


if product_frames:

    product_all = pd.concat(
        product_frames,
        ignore_index=True,
        sort=False
    )

else:

    product_all = pd.DataFrame()


print(
    f"통합 전 제품분석: "
    f"{len(product_all):,} rows"
)


# ================================================================================
# 6. ASIN별 검색키워드 관계 분석
# ================================================================================

asin_keyword_map = {}


if not product_all.empty:

    valid = product_all[

        product_all["ASIN"]
        .astype(str)
        .str.strip()
        .ne("")

    ]


    for asin, group in valid.groupby(
        "ASIN"
    ):

        keywords = (

            group["검색키워드"]

            .dropna()

            .apply(
                normalize_keyword
            )

            .tolist()

        )


        representative, all_keywords = (

            choose_representative_keyword(
                keywords
            )

        )


        asin_keyword_map[asin] = {

            "대표": representative,

            "전체": all_keywords

        }


# ================================================================================
# 7. 제품분석 대표 행 선택
# ================================================================================

if not product_all.empty:

    product_all[
        "_rank_numeric"
    ] = pd.to_numeric(

        product_all.get(

            "검색순위",

            pd.Series(
                index=product_all.index
            )

        ),

        errors="coerce"

    )


    has_asin = (

        product_all["ASIN"]
        .astype(str)
        .str.strip()
        .ne("")

    )


    product_with_asin = product_all[
        has_asin
    ].copy()


    product_without_asin = product_all[
        ~has_asin
    ].copy()


    # --------------------------------------------------------------------------
    # 검색어별 대표 행 우선순위
    #
    # hierarchy:
    #
    # patch
    # └── skin patch
    #     ├── acne patch
    #     ├── wrinkle patch
    #     └── eye patch
    #
    # skin patch를 가장 우선.
    # 세부 검색어끼리는 동급.
    # --------------------------------------------------------------------------

    def row_keyword_priority(keyword):

        keyword = normalize_keyword(
            keyword
        )


        if keyword == "skin patch":

            return 4


        if keyword in [

            "acne patch",
            "wrinkle patch",
            "eye patch"

        ]:

            return 3


        if keyword == "patch":

            return 2


        return 1


    product_with_asin[
        "_keyword_priority"
    ] = (

        product_with_asin[
            "검색키워드"
        ]

        .apply(
            row_keyword_priority
        )

    )


    product_with_asin = (

        product_with_asin

        .sort_values(

            by=[
                "ASIN",
                "_keyword_priority",
                "_rank_numeric"
            ],

            ascending=[
                True,
                False,
                True
            ],

            na_position="last"

        )

    )


    # --------------------------------------------------------------------------
    # ASIN당 대표 행 1개
    # --------------------------------------------------------------------------

    product_unique = (

        product_with_asin

        .drop_duplicates(

            subset=[
                "ASIN"
            ],

            keep="first"

        )

        .copy()

    )


    # --------------------------------------------------------------------------
    # 대표 검색키워드
    # --------------------------------------------------------------------------

    product_unique[
        "검색키워드"
    ] = product_unique[
        "ASIN"
    ].map(

        lambda x:

            asin_keyword_map
            .get(x, {})
            .get(
                "대표",
                ""
            )

    )


    # --------------------------------------------------------------------------
    # 검색키워드 전체
    # --------------------------------------------------------------------------

    product_unique[
        "검색키워드_전체"
    ] = product_unique[
        "ASIN"
    ].map(

        lambda x:

            " | ".join(

                asin_keyword_map
                .get(x, {})
                .get(
                    "전체",
                    []
                )

            )

    )


    # --------------------------------------------------------------------------
    # ASIN 없는 행 유지
    # --------------------------------------------------------------------------

    product_final = pd.concat(

        [

            product_unique,

            product_without_asin

        ],

        ignore_index=True,

        sort=False

    )

else:

    product_final = pd.DataFrame()


# ================================================================================
# 8. 제품분석 컬럼 정리
# ================================================================================

product_final = product_final.drop(

    columns=[

        "_source_keyword",
        "_keyword_priority",
        "_rank_numeric"

    ],

    errors="ignore"

)


preferred_product_cols = [

    "검색순위",
    "검색키워드",
    "검색키워드_전체",
    "브랜드",
    "상품명",
    "상품URL",
    "ASIN",
    "판매가격",
    "정가",
    "할인율",
    "평점",
    "리뷰수",
    "페이지",

]


existing = [

    c

    for c in preferred_product_cols

    if c in product_final.columns

]


remaining = [

    c

    for c in product_final.columns

    if c not in existing

]


product_final = product_final[
    existing + remaining
]

print(
    f"제품분석 최종: "
    f"{len(product_final):,} rows"
)


print(
    f"고유 ASIN: "
    f"{product_final['ASIN'].replace('', np.nan).dropna().nunique():,}"
)


# ================================================================================
# 9. 가격요약 통합
# ================================================================================

print("\n" + "=" * 100)
print("② 가격요약 통합")
print("=" * 100)


summary_frames = []


for keyword, sheets in data.items():

    if "가격요약" not in sheets:

        continue


    df = sheets[
        "가격요약"
    ].copy()


    df = ensure_column(
        df,
        "검색키워드"
    )


    df = ensure_column(
        df,
        "ASIN"
    )


    df["검색키워드"] = (

        df["검색키워드"]

        .replace(
            [
                "",
                "nan",
                "None"
            ],
            np.nan
        )

        .fillna(
            keyword
        )

        .apply(
            normalize_keyword
        )

    )


    df["ASIN"] = (

        df["ASIN"]

        .apply(
            normalize_asin
        )

    )


    summary_frames.append(
        df
    )


if summary_frames:

    summary_all = pd.concat(

        summary_frames,

        ignore_index=True,

        sort=False

    )

else:

    summary_all = pd.DataFrame()


print(
    f"가격요약 통합 전: "
    f"{len(summary_all):,} rows"
)


# ================================================================================
# 10. 가격요약 ASIN 중복 제거
# ================================================================================

if not summary_all.empty:

    has_asin = (

        summary_all["ASIN"]
        .astype(str)
        .str.strip()
        .ne("")

    )


    summary_with_asin = summary_all[
        has_asin
    ].copy()


    summary_without_asin = summary_all[
        ~has_asin
    ].copy()


    # --------------------------------------------------------------------------
    # 가격요약 대표 행 우선순위
    #
    # skin patch = 상위 범주
    # acne / wrinkle / eye = 동일한 세부 범주
    # patch = 전체 범주
    # --------------------------------------------------------------------------

    def summary_keyword_priority(keyword):

        keyword = normalize_keyword(
            keyword
        )


        if keyword == "skin patch":

            return 4


        if keyword in [

            "acne patch",
            "wrinkle patch",
            "eye patch"

        ]:

            return 3


        if keyword == "patch":

            return 2


        return 1


    summary_with_asin[
        "_keyword_priority"
    ] = (

        summary_with_asin[
            "검색키워드"
        ]

        .apply(
            summary_keyword_priority
        )

    )


    summary_with_asin = (

        summary_with_asin

        .sort_values(

            by=[
                "ASIN",
                "_keyword_priority"
            ],

            ascending=[
                True,
                False
            ]

        )

    )


    summary_unique = (

        summary_with_asin

        .drop_duplicates(

            subset=[
                "ASIN"
            ],

            keep="first"

        )

        .copy()

    )


    # --------------------------------------------------------------------------
    # 대표 검색키워드
    # --------------------------------------------------------------------------

    summary_unique[
        "검색키워드"
    ] = summary_unique[
        "ASIN"
    ].map(

        lambda x:

            asin_keyword_map
            .get(x, {})
            .get(
                "대표",
                ""
            )

    )


    # --------------------------------------------------------------------------
    # 전체 검색키워드
    # --------------------------------------------------------------------------

    summary_unique[
        "검색키워드_전체"
    ] = summary_unique[
        "ASIN"
    ].map(

        lambda x:

            " | ".join(

                asin_keyword_map
                .get(x, {})
                .get(
                    "전체",
                    []
                )

            )

    )


    summary_final = pd.concat(

        [

            summary_unique,

            summary_without_asin

        ],

        ignore_index=True,

        sort=False

    )

else:

    summary_final = pd.DataFrame()


summary_final = summary_final.drop(

    columns=[

        "_keyword_priority"

    ],

    errors="ignore"

)


# ================================================================================
# 11. 가격이력 통합
# ================================================================================

print("\n" + "=" * 100)
print("③ 가격이력 통합")
print("=" * 100)


history_frames = []


for keyword, sheets in data.items():

    if "가격이력" not in sheets:

        continue


    df = sheets[
        "가격이력"
    ].copy()


    df = ensure_column(
        df,
        "검색키워드"
    )


    df = ensure_column(
        df,
        "ASIN"
    )


    df = ensure_column(
        df,
        "날짜"
    )


    df["검색키워드"] = (

        df["검색키워드"]

        .replace(
            [
                "",
                "nan",
                "None"
            ],
            np.nan
        )

        .fillna(
            keyword
        )

        .apply(
            normalize_keyword
        )

    )


    df["ASIN"] = (

        df["ASIN"]

        .apply(
            normalize_asin
        )

    )


    df["날짜"] = pd.to_datetime(

        df["날짜"],

        errors="coerce"

    )


    history_frames.append(
        df
    )


    print(

        f"{keyword:<20} "
        f"{len(df):>8,} rows"

    )


if history_frames:

    history_all = pd.concat(

        history_frames,

        ignore_index=True,

        sort=False

    )

else:

    history_all = pd.DataFrame()


print(
    f"\n가격이력 전체: "
    f"{len(history_all):,} rows"
)


# ================================================================================
# 12. 가격이력 중복 제거
# ================================================================================

if not history_all.empty:

    # --------------------------------------------------------------------------
    # 중요
    #
    # ASIN만으로 중복 제거하지 않는다.
    #
    # 같은 ASIN의 날짜별 가격은 모두 유지.
    #
    # 동일 ASIN + 동일 timestamp + 동일 가격만 중복 제거
    # --------------------------------------------------------------------------

    history_all[
        "_date_key"
    ] = history_all[
        "날짜"
    ].dt.strftime(
        "%Y-%m-%d %H:%M:%S"
    )


    history_all[
        "_price_key"
    ] = history_all[
        "가격"
    ].astype(str).str.strip()


    history_final = (

        history_all

        .drop_duplicates(

            subset=[

                "ASIN",
                "_date_key",
                "_price_key"

            ],

            keep="first"

        )

        .copy()

    )


    # --------------------------------------------------------------------------
    # 날짜순 정렬
    # --------------------------------------------------------------------------

    history_final = (

        history_final

        .sort_values(

            by=[
                "ASIN",
                "날짜"
            ],

            ascending=[
                True,
                True
            ],

            na_position="last"

        )

    )


    # --------------------------------------------------------------------------
    # 임시 컬럼 삭제
    # --------------------------------------------------------------------------

    history_final = history_final.drop(

        columns=[

            "_date_key",
            "_price_key"

        ],

        errors="ignore"

    )

else:

    history_final = pd.DataFrame()


# ================================================================================
# 13. 가격이력 검증
# ================================================================================

print("\n" + "=" * 100)
print("가격이력 검증")
print("=" * 100)


if not history_final.empty:

    print(

        "고유 ASIN:",

        history_final[
            "ASIN"
        ]
        .replace(
            "",
            np.nan
        )
        .dropna()
        .nunique()

    )


    print(

        "전체 가격이력:",

        len(history_final)

    )


    print(

        "날짜:",

        history_final[
            "날짜"
        ].min(),

        "~",

        history_final[
            "날짜"
        ].max()

    )


    # --------------------------------------------------------------------------
    # ASIN별 가격이력 건수
    # --------------------------------------------------------------------------

    history_counts = (

        history_final

        .groupby(
            "ASIN"
        )

        .size()

        .sort_values(
            ascending=False
        )

    )


    print(
        "\n가격이력 건수 TOP 10:"
    )


    print(
        history_counts.head(10)
    )


# ================================================================================
# 14. 중복 검색어 ASIN 검증
# ================================================================================

print("\n" + "=" * 100)
print("검색어 중복 ASIN 검증")
print("=" * 100)


if asin_keyword_map:

    multi_keyword = {

        asin: info

        for asin, info
        in asin_keyword_map.items()

        if len(info["전체"]) > 1

    }


    print(

        "여러 검색어에서 발견된 ASIN:",

        len(multi_keyword)

    )


    print("\n예시:")


    for i, (
        asin,
        info
    ) in enumerate(
        multi_keyword.items()
    ):

        if i >= 20:

            break


        print(

            f"{asin}: "
            f"대표 = {info['대표']} | "
            f"전체 = {' | '.join(info['전체'])}"

        )


# ================================================================================
# 15. Excel 저장
# ================================================================================

print("\n" + "=" * 100)
print("Excel 저장")
print("=" * 100)


with pd.ExcelWriter(

    output_file,

    engine="openpyxl"

) as writer:

    product_final.to_excel(

        writer,

        sheet_name="제품분석",

        index=False

    )


    summary_final.to_excel(

        writer,

        sheet_name="가격요약",

        index=False

    )


    history_final.to_excel(

        writer,

        sheet_name="가격이력",

        index=False

    )


# ================================================================================
# 16. 최종 결과
# ================================================================================

print("\n" + "=" * 100)
print("✅ 통합 완료")
print("=" * 100)


print(
    f"""
파일:
{output_file}

제품분석:
{len(product_final):,} rows

가격요약:
{len(summary_final):,} rows

가격이력:
{len(history_final):,} rows
"""
)


# ================================================================================
# 17. 통합 Excel 자동 다운로드
# ================================================================================

from google.colab import files
import glob


print("\n" + "=" * 100)
print("📥 통합 Excel 자동 다운로드")
print("=" * 100)


# Amazon Patch 통합 결과 파일 자동 탐색

output_candidates = glob.glob(

    "/content/Amazon_patch_5keyword_append_*.xlsx"

)


if not output_candidates:

    print(
        "❌ 통합 Excel 파일을 찾을 수 없습니다."
    )

else:

    # 가장 최근에 생성된 파일 선택

    output_file = max(

        output_candidates,

        key=os.path.getmtime

    )


    print(
        "다운로드 파일:"
    )

    print(
        output_file
    )


    print(
        "\n📥 다운로드를 시작합니다..."
    )


    files.download(
        output_file
    )


    print(
        "✅ 다운로드 요청 완료"
    )